# KoBERT 기반 2차 모델 학습 (순수 텍스트 기반)
## 맥락 기반 보이스피싱 탐지 모델 - 화자 정보 제거

## 1. 환경 설정 및 라이브러리 import

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

d:\workspace\woogawooga_project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## 2. 데이터 로딩 및 전처리

In [2]:
# 훈련 데이터 로딩 (merged_labeled_data.csv)
train_df = pd.read_csv("../../dataset/merged_labeled_data.csv")
print(f"훈련 데이터 수: {len(train_df)}")
print(f"피싱: {train_df['is_phishing'].sum()}, 일반: {len(train_df) - train_df['is_phishing'].sum()}")
print(f"고유 file_id 수: {train_df['file_id'].nunique()}")

# 테스트 데이터 로딩 (1차모델_테스트데이터셋.csv)
test_df = pd.read_csv("../../dataset/1차모델_테스트데이터셋.csv")
print(f"\n테스트 데이터 수: {len(test_df)}")
print(f"피싱: {test_df['is_phishing'].sum()}, 일반: {len(test_df) - test_df['is_phishing'].sum()}")

# 데이터 미리보기
print("\n=== 훈련 데이터 미리보기 ===")
print(train_df.head())
print("\n=== 테스트 데이터 미리보기 ===")
print(test_df.head())

훈련 데이터 수: 8047
피싱: 2047, 일반: 6000
고유 file_id 수: 8047

테스트 데이터 수: 1000
피싱: 500, 일반: 500

=== 훈련 데이터 미리보기 ===
        file_id phishing_type  \
0  phishing_000       가족지인사칭형   
1  phishing_001       가족지인사칭형   
2  phishing_002       가족지인사칭형   
3  phishing_003       가족지인사칭형   
4  phishing_004       가족지인사칭형   

                                                text  is_phishing  
0   여보세요, OOO야. 나 아빠야. 아빠? 목소리가 좀 이상한데요. 아빠 핸드폰이 ...            1  
1   OOO야, 나야. 너 친구 OOO인데, 전화번호 바뀌었어. 어? 갑자기 왜 바꿨어...            1  
2   여보세요, OOO? 나 엄마 친구야.  아, 네. 무슨 일이세요? 네 엄마가 지금...            1  
3   OOO야, 나야. 오랜만이다. 오랜만이네. 근데 네 목소리 왜 그래? 나 사고가 ...            1  
4   여보세요, OOO님? 여기 경찰서입니다. 경찰서요? 무슨 일이죠? 네 아들이 교통...            1  

=== 테스트 데이터 미리보기 ===
  file_name                                               text  is_phishing
0         0  예 고객님 담당자 김성도 대리입니다.예지금 법무사님이 두분 배정되셨어요.네 네네 네...            1
1         2  6시 되가지고 전화 해봤습니다.예 예 그 앞전에 이면주 법무사님 영수증 확인되셨는데...            1
2         3  네 네 네 여보세요네 네어디에 계시는 겁

In [3]:
# 대화 시퀀스 생성
def create_dialogue_sequences_from_merged(df):
    dialogues = []
    
    for file_id in df['file_id'].unique():
        file_data = df[df['file_id'] == file_id]
        full_text = file_data['text'].iloc[0]
        sentences = [s.strip() for s in full_text.split('.') if s.strip()]
        label = file_data['is_phishing'].iloc[0]
        
        if len(sentences) > 1:
            dialogues.append({
                'file_id': file_id,
                'texts': sentences,
                'label': label
            })
    
    return dialogues

def create_single_text_data(df):
    test_data = []
    
    for idx, row in df.iterrows():
        full_text = row['text']
        sentences = [s.strip() for s in full_text.split('.') if s.strip()]
        
        if len(sentences) == 0:
            sentences = [full_text]
        
        test_data.append({
            'file_id': f"test_{idx}",
            'texts': sentences,
            'label': row['is_phishing']
        })
    
    return test_data

train_dialogues = create_dialogue_sequences_from_merged(train_df)
test_dialogues = create_single_text_data(test_df)

print(f"생성된 훈련 대화 시퀀스 수: {len(train_dialogues)}")
print(f"평균 문장 수: {np.mean([len(d['texts']) for d in train_dialogues]):.2f}")
print(f"생성된 테스트 데이터 수: {len(test_dialogues)}")

생성된 훈련 대화 시퀀스 수: 7825
평균 문장 수: 29.06
생성된 테스트 데이터 수: 1000


## 3. KoBERT 및 데이터셋 클래스 정의

In [4]:
MODEL_NAME = "skt/kobert-base-v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
kobert_model = AutoModel.from_pretrained(MODEL_NAME)

class TextOnlyDialogueDataset(Dataset):
    def __init__(self, dialogues, tokenizer, max_length=128, max_turns=50):
        self.dialogues = dialogues
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_turns = max_turns
    
    def __len__(self):
        return len(self.dialogues)
    
    def __getitem__(self, idx):
        dialogue = self.dialogues[idx]
        texts = dialogue['texts'][:self.max_turns]
        label = dialogue['label']
        
        input_ids_list = []
        attention_mask_list = []
        
        for text in texts:
            text = str(text).strip()
            if len(text) == 0:
                text = "[EMPTY]"
            
            encoded = self.tokenizer(
                text,
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            
            input_ids_list.append(encoded['input_ids'].squeeze(0))
            attention_mask_list.append(encoded['attention_mask'].squeeze(0))
        
        return {
            'input_ids': torch.stack(input_ids_list),
            'attention_mask': torch.stack(attention_mask_list),
            'label': torch.tensor(label, dtype=torch.long),
            'num_turns': len(texts)
        }

def collate_fn_text_only(batch):
    max_turns = max([item['num_turns'] for item in batch])
    
    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []
    batch_lengths = []
    
    for item in batch:
        num_turns = item['num_turns']
        
        if num_turns < max_turns:
            pad_size = max_turns - num_turns
            pad_input_ids = torch.zeros(pad_size, item['input_ids'].size(1), dtype=torch.long)
            pad_attention_mask = torch.zeros(pad_size, item['attention_mask'].size(1), dtype=torch.long)
            
            input_ids = torch.cat([item['input_ids'], pad_input_ids], dim=0)
            attention_mask = torch.cat([item['attention_mask'], pad_attention_mask], dim=0)
        else:
            input_ids = item['input_ids']
            attention_mask = item['attention_mask']
        
        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(item['label'])
        batch_lengths.append(num_turns)
    
    return {
        'input_ids': torch.stack(batch_input_ids),
        'attention_mask': torch.stack(batch_attention_mask),
        'labels': torch.stack(batch_labels),
        'lengths': torch.tensor(batch_lengths, dtype=torch.long)
    }

print("텍스트 전용 데이터셋 클래스 정의 완료")

텍스트 전용 데이터셋 클래스 정의 완료


## 4. 텍스트 전용 모델 정의

In [5]:
class TextOnlyPhishingDetector(nn.Module):
    def __init__(self, kobert_model, hidden_size=256, num_classes=2, dropout=0.3):
        super(TextOnlyPhishingDetector, self).__init__()
        
        self.kobert = kobert_model
        self.kobert_hidden_size = kobert_model.config.hidden_size
        
        self.sentence_projection = nn.Linear(
            self.kobert_hidden_size,
            hidden_size
        )
        
        self.dialogue_lstm = nn.LSTM(
            hidden_size, 
            hidden_size // 2, 
            batch_first=True, 
            bidirectional=True,
            dropout=dropout if hidden_size > 1 else 0
        )
        
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_classes)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        if isinstance(self.sentence_projection, nn.Linear):
            nn.init.xavier_uniform_(self.sentence_projection.weight)
            nn.init.zeros_(self.sentence_projection.bias)
        
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)
    
    def forward(self, input_ids, attention_mask, lengths):
        batch_size, max_turns, seq_len = input_ids.size()
        
        input_ids_flat = input_ids.view(-1, seq_len)
        attention_mask_flat = attention_mask.view(-1, seq_len)
        
        with torch.no_grad():
            kobert_outputs = self.kobert(
                input_ids=input_ids_flat,
                attention_mask=attention_mask_flat
            )
        
        sentence_embeddings = kobert_outputs.last_hidden_state[:, 0, :]
        sentence_embeddings = sentence_embeddings.view(batch_size, max_turns, -1)
        
        sentence_features = self.sentence_projection(sentence_embeddings)
        
        max_len = max_turns
        padding_mask = torch.arange(max_len, device=lengths.device).expand(
            batch_size, max_len
        ) >= lengths.unsqueeze(1)
        
        lstm_out, _ = self.dialogue_lstm(sentence_features)
        
        attended_out, attention_weights = self.attention(
            lstm_out, lstm_out, lstm_out,
            key_padding_mask=padding_mask
        )
        
        mask = ~padding_mask.unsqueeze(-1)
        masked_attended = attended_out * mask
        dialogue_repr = masked_attended.sum(dim=1) / lengths.unsqueeze(-1).float()
        
        logits = self.classifier(dialogue_repr)
        
        return {
            'logits': logits,
            'attention_weights': attention_weights,
            'dialogue_repr': dialogue_repr
        }

model = TextOnlyPhishingDetector(kobert_model)
model.to(device)

print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print("화자 임베딩 제거된 텍스트 전용 모델 초기화 완료")

모델 파라미터 수: 93,075,330
화자 임베딩 제거된 텍스트 전용 모델 초기화 완료


## 5. K-Fold 설정 및 훈련 함수

In [6]:
file_ids = [d['file_id'] for d in train_dialogues]
labels = [d['label'] for d in train_dialogues]

test_dataset = TextOnlyDialogueDataset(test_dialogues, tokenizer)

K_FOLDS = 5
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
BATCH_SIZE = 8

print(f"훈련 대화 수: {len(train_dialogues)}")
print(f"테스트 데이터: {len(test_dialogues)}")
print(f"K-Fold 수: {K_FOLDS}")

훈련 대화 수: 7825
테스트 데이터: 1000
K-Fold 수: 5


In [7]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        lengths = batch['lengths'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, lengths)
        logits = outputs['logits']
        
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(train_loader), 100. * correct / total

def validate_epoch(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            lengths = batch['lengths'].to(device)
            
            outputs = model(input_ids, attention_mask, lengths)
            logits = outputs['logits']
            
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(val_loader), 100. * correct / total, all_predictions, all_labels

print("훈련 및 검증 함수 정의 완료")

훈련 및 검증 함수 정의 완료


In [8]:
# 클래스 가중치 계산 함수
def calculate_class_weights(labels):
    """
    데이터 불균형을 해결하기 위한 클래스 가중치 계산
    """
    from sklearn.utils.class_weight import compute_class_weight
    import numpy as np
    
    unique_classes = np.unique(labels)
    class_weights = compute_class_weight(
        'balanced', 
        classes=unique_classes, 
        y=labels
    )
    
    class_weight_dict = dict(zip(unique_classes, class_weights))
    print(f"클래스 분포: {np.bincount(labels)}")
    print(f"클래스 가중치: {class_weight_dict}")
    
    # PyTorch tensor로 변환
    weight_tensor = torch.FloatTensor([class_weights[0], class_weights[1]])
    return weight_tensor

# 전체 훈련 데이터의 클래스 가중치 계산
all_labels = np.array(labels)
class_weights = calculate_class_weights(all_labels)

print(f"적용될 클래스 가중치: Normal={class_weights[0]:.3f}, Phishing={class_weights[1]:.3f}")

print("K-Fold 교차검증 시작...")

fold_results = []
all_val_accs = []

for fold, (train_indices, val_indices) in enumerate(skf.split(file_ids, labels)):
    print(f"\n{'='*50}")
    print(f"Fold {fold+1}/{K_FOLDS} 시작")
    print(f"{'='*50}")
    
    fold_train_dialogues = [train_dialogues[i] for i in train_indices]
    fold_val_dialogues = [train_dialogues[i] for i in val_indices]
    
    print(f"Fold {fold+1} - 훈련: {len(fold_train_dialogues)}, 검증: {len(fold_val_dialogues)}")
    
    # 현재 fold의 클래스 분포 확인 및 가중치 계산
    fold_train_labels = [fold_train_dialogues[i]['label'] for i in range(len(fold_train_dialogues))]
    fold_class_weights = calculate_class_weights(np.array(fold_train_labels))
    
    fold_train_dataset = TextOnlyDialogueDataset(fold_train_dialogues, tokenizer)
    fold_val_dataset = TextOnlyDialogueDataset(fold_val_dialogues, tokenizer)
    
    fold_train_loader = DataLoader(
        fold_train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn_text_only
    )
    fold_val_loader = DataLoader(
        fold_val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_text_only
    )
    
    fold_model = TextOnlyPhishingDetector(kobert_model).to(device)
    
    # 클래스 가중치를 적용한 손실 함수
    criterion = nn.CrossEntropyLoss(weight=fold_class_weights.to(device))
    
    optimizer = optim.AdamW(fold_model.parameters(), lr=2e-5, weight_decay=0.01)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )
    
    fold_train_accs = []
    fold_val_accs = []
    best_val_acc = 0
    
    NUM_EPOCHS = 8
    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_epoch(fold_model, fold_train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_preds, val_labels = validate_epoch(fold_model, fold_val_loader, criterion, device)
        
        scheduler.step(val_loss)
        
        fold_train_accs.append(train_acc)
        fold_val_accs.append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        
        if epoch % 2 == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%")
    
    print(f"Fold {fold+1} 최고 검증 정확도: {best_val_acc:.2f}%")
    
    fold_results.append({
        'fold': fold+1,
        'best_val_acc': best_val_acc,
        'final_val_acc': fold_val_accs[-1],
        'train_accs': fold_train_accs,
        'val_accs': fold_val_accs,
        'val_predictions': val_preds,
        'val_labels': val_labels
    })
    all_val_accs.append(best_val_acc)

# K-Fold 결과 요약
print(f"\n{'='*60}")
print("K-Fold 교차검증 결과 요약")
print(f"{'='*60}")

for i, result in enumerate(fold_results):
    print(f"Fold {i+1}: {result['best_val_acc']:.2f}%")

mean_acc = np.mean(all_val_accs)
std_acc = np.std(all_val_accs)

print(f"\n평균 검증 정확도: {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"최고 검증 정확도: {max(all_val_accs):.2f}%")
print(f"최저 검증 정확도: {min(all_val_accs):.2f}%")

클래스 분포: [5797 2028]
클래스 가중치: {np.int64(0): np.float64(0.6749180610660687), np.int64(1): np.float64(1.929240631163708)}
적용될 클래스 가중치: Normal=0.675, Phishing=1.929
K-Fold 교차검증 시작...

Fold 1/5 시작
Fold 1 - 훈련: 6260, 검증: 1565
클래스 분포: [4638 1622]
클래스 가중치: {np.int64(0): np.float64(0.6748598533850798), np.int64(1): np.float64(1.9297163995067816)}


KeyboardInterrupt: 

In [ ]:
print(f"\n{'='*50}")
print("전체 데이터로 최종 모델 훈련")
print(f"{'='*50}")

final_train_dataset = TextOnlyDialogueDataset(train_dialogues, tokenizer)
final_train_loader = DataLoader(
    final_train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn_text_only
)

final_model = TextOnlyPhishingDetector(kobert_model).to(device)

# 전체 데이터의 클래스 가중치 적용
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

optimizer = optim.AdamW(final_model.parameters(), lr=2e-5, weight_decay=0.01)

NUM_FINAL_EPOCHS = 8
for epoch in range(NUM_FINAL_EPOCHS):
    train_loss, train_acc = train_epoch(final_model, final_train_loader, criterion, optimizer, device)
    if epoch % 2 == 0:
        print(f"Final Epoch {epoch+1}/{NUM_FINAL_EPOCHS} - Train Acc: {train_acc:.2f}%")

# 테스트 데이터 평가
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_text_only
)

print(f"\n테스트 데이터 평가 시작... (테스트 샘플 수: {len(test_dialogues)})")

final_model.eval()
test_correct = 0
test_total = 0
test_predictions = []
test_labels = []
test_probabilities = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        lengths = batch['lengths'].to(device)
        
        outputs = final_model(input_ids, attention_mask, lengths)
        logits = outputs['logits']
        
        probs = torch.softmax(logits, dim=1)
        
        _, predicted = torch.max(logits.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        
        test_predictions.extend(predicted.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())
        test_probabilities.extend(probs.cpu().numpy())

test_accuracy = 100. * test_correct / test_total
print(f"\n=== 최종 테스트 결과 ===")
print(f"K-Fold 평균 검증 정확도: {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"테스트 정확도: {test_accuracy:.2f}%")
print("\n테스트 데이터 분류 리포트:")
print(classification_report(test_labels, test_predictions, target_names=['Normal', 'Phishing']))

# 클래스별 성능 분석
from sklearn.metrics import precision_recall_fscore_support
precision, recall, f1, support = precision_recall_fscore_support(test_labels, test_predictions)
print(f"\n=== 클래스별 상세 성능 (불균형 데이터 대응) ===")
print(f"Normal   - Precision: {precision[0]:.3f}, Recall: {recall[0]:.3f}, F1: {f1[0]:.3f}")
print(f"Phishing - Precision: {precision[1]:.3f}, Recall: {recall[1]:.3f}, F1: {f1[1]:.3f}")

# ROC AUC 계산
test_probs = np.array(test_probabilities)[:, 1]
fpr, tpr, _ = roc_curve(test_labels, test_probs)
roc_auc = auc(fpr, tpr)
print(f"\n테스트 AUC: {roc_auc:.3f}")

# 데이터 불균형 대응 효과 분석
print(f"\n=== 데이터 불균형 대응 효과 ===")
print(f"적용된 클래스 가중치 - Normal: {class_weights[0]:.3f}, Phishing: {class_weights[1]:.3f}")
print(f"테스트 데이터 클래스 분포: {np.bincount(test_labels)}")
print(f"Balanced Accuracy: {np.mean([recall[0], recall[1]]):.3f}")

# 결과 시각화

In [ ]:
# 결과 시각화
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Fold별 성능 비교

In [ ]:
# Fold별 성능 비교
folds = [f"Fold {i+1}" for i in range(K_FOLDS)]
ax1.bar(folds, all_val_accs, alpha=0.7, color="skyblue")
ax1.axhline(y=mean_acc, color="red", linestyle="--", label=f"평균: {mean_acc:.2f}%")
ax1.set_title("Fold별 검증 정확도 (클래스 가중치 적용)")
ax1.set_xlabel("Fold")
ax1.set_ylabel("Accuracy (%)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# 테스트 confusion matrix

In [ ]:
# 테스트 confusion matrix
cm = confusion_matrix(test_labels, test_predictions)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=ax2,
    xticklabels=["Normal", "Phishing"],
    yticklabels=["Normal", "Phishing"],
)
ax2.set_title(f"Test Confusion Matrix (Accuracy: {test_accuracy:.2f}%)")
ax2.set_xlabel("Predicted")
ax2.set_ylabel("Actual")

# ROC Curve

In [ ]:
# ROC Curve
ax3.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
ax3.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
ax3.set_xlim([0.0, 1.0])
ax3.set_ylim([0.0, 1.05])
ax3.set_xlabel("False Positive Rate")
ax3.set_ylabel("True Positive Rate")
ax3.set_title("ROC Curve")
ax3.legend(loc="lower right")
ax3.grid(True, alpha=0.3)

# 클래스별 성능 비교 (Precision, Recall, F1)

In [ ]:
# 클래스별 성능 비교 (Precision, Recall, F1)
metrics = ["Precision", "Recall", "F1-Score"]
normal_scores = [precision[0], recall[0], f1[0]]
phishing_scores = [precision[1], recall[1], f1[1]]

x = np.arange(len(metrics))
width = 0.35

ax4.bar(x - width / 2, normal_scores, width, label="Normal", alpha=0.7, color="blue")
ax4.bar(x + width / 2, phishing_scores, width, label="Phishing", alpha=0.7, color="red")
ax4.set_xlabel("Metrics")
ax4.set_ylabel("Score")
ax4.set_title("클래스별 성능 지표 (불균형 대응)")
ax4.set_xticks(x)
ax4.set_xticklabels(metrics)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    "../../datas/analysisData/2nd_model_balanced_results.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# 최종 모델 저장

In [ ]:
# 최종 모델 저장
final_model_path = "../../models/kobert_2nd_model_balanced.pth"
torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "model_config": {"hidden_size": 256, "num_classes": 2, "dropout": 0.3},
        "tokenizer_name": MODEL_NAME,
        "class_weights": class_weights.tolist(),
        "kfold_results": {
            "mean_val_acc": mean_acc,
            "std_val_acc": std_acc,
            "fold_accs": all_val_accs,
            "fold_details": fold_results,
        },
        "test_accuracy": test_accuracy,
        "test_auc": roc_auc,
        "balanced_accuracy": np.mean([recall[0], recall[1]]),
    },
    final_model_path,
)

print(f"\n최종 모델이 저장되었습니다: {final_model_path}")

# 결과 요약

In [ ]:
# 결과 요약
print("\n" + "=" * 60)
print("2차 모델 (클래스 불균형 대응) 완료 요약")
print("=" * 60)
print(f"모델 아키텍처: KoBERT + LSTM + Attention (화자 정보 제거)")
print(f"교차검증: {K_FOLDS}-Fold Stratified")
print(f"훈련 데이터: {len(train_dialogues)} 대화 (merged_labeled_data.csv)")
print(f"테스트 데이터: {len(test_dialogues)} 샘플 (1차모델_테스트데이터셋.csv)")
print(f"")
print(f"=== 불균형 데이터 대응 ===")
print(f"클래스 가중치: Normal={class_weights[0]:.3f}, Phishing={class_weights[1]:.3f}")
print(f"")
print(f"=== 성능 결과 ===")
print(f"K-Fold 평균 검증 정확도: {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"테스트 정확도: {test_accuracy:.2f}%")
print(f"테스트 AUC: {roc_auc:.3f}")
print(f"Balanced Accuracy: {np.mean([recall[0], recall[1]]):.3f}")
print(f"")
print(f"=== 클래스별 성능 ===")
print(
    f"Normal   - Precision: {precision[0]:.3f}, Recall: {recall[0]:.3f}, F1: {f1[0]:.3f}"
)
print(
    f"Phishing - Precision: {precision[1]:.3f}, Recall: {recall[1]:.3f}, F1: {f1[1]:.3f}"
)
print(f"")
print(
    f"성능 안정성: 표준편차 {std_acc:.2f}%로 {'안정적' if std_acc < 2.0 else '다소 불안정'}"
)
print("클래스 불균형 대응으로 피싱 탐지 성능 향상!")
print("=" * 60)